# Additional Test 2 - second-VQA-model robustness (the last big gap)

Every selective-prediction conclusion in the paper currently rests on **one**
frozen VQA model (ViLT, 18.5% VizWiz accuracy). The most likely reviewer
objection is: *"confidence dominates and defects add nothing" may be an
artifact of a weak model.* This notebook closes that gap with a second frozen
VQA model of a different architecture AND confidence type: **BLIP-VQA-base**,
generative, with length-normalized sequence probability as confidence.

| Cell | What it adds to the paper | Runtime | Wall-clock (first run) |
|------|---------------------------|---------|------------------------|
| SETUP | clone/pull + missing deps only | any | 1-3 min |
| E6b | BLIP answer + confidence harvest over all 24k pairs (cached to Drive, shard-resumable) | **GPU (A100 fastest, L4 fine)** | A100 ~25-40 min, L4 ~60-90 min |
| E7d | Full diagnostic battery vs BLIP confidence: defect-conditioned calibration, learned conf+defects, triage alone, conf+triage, full stack, GT-defect oracle + BH-FDR + gated ARR/FRR at 90% coverage | CPU (can run in the same GPU session) | ~20-30 min (bootstrap) |
| E8d | Regenerate F8 with the full method battery; adds the BLIP panel automatically after E7d | CPU | <1 min |

**Prerequisite:** run the REFRESH cell in `followup_experiments.ipynb` first if
you haven't - E7d's comparisons should sit next to snapshot-consistent
E7/E7b numbers.

**Interpretation guide for E7d:**
- If no positive improvement survives BH-FDR on BLIP either, the paper's
  claim upgrades to: *"across two frozen VQA models with different
  architectures and confidence types"* - the negative result becomes a
  pattern, not an anecdote.
- If BLIP behaves differently (e.g., defects DO help a stronger model's
  confidence), that is an even more interesting finding - the paper gains a
  model-dependence analysis instead. Either outcome strengthens the paper.

When done, download `results/E6b_vqaconf_blip/` (JSON only, skip the parquet
if large), `results/E7d_blip_diagnostics/`, and `results/figures/F8_*`, then
paste the E7d summary printout here for the manuscript update.


## SETUP - clone repo + minimal deps *(run first on every fresh runtime)*

In [ ]:
# ====================================================================
#  SETUP - run FIRST on every fresh runtime (any runtime type).
#  Clones/updates the repo and installs ONLY missing packages.
#  Never reinstalls or downgrades anything Colab already ships
#  (that is what used to break the NumPy/pandas binary stack).
#  Re-running on a warm runtime finishes in seconds.
# ====================================================================
import importlib.util, os, subprocess, sys

REPO_URL  = 'https://github.com/meteorboyF/VQA-paper.git'
REPO_ROOT = '/content/VQA-paper'

# ── Optional switches (set BEFORE anything imports src) ──────────────
# os.environ['VQA_FORCE_RERUN'] = '1'                      # ignore all caches/DONE markers
# os.environ['VQA_BACKBONES']   = 'clip,mobilenet,dinov2'  # full 3-backbone table (A100 recommended)
# os.environ['VQA_DRIVE_BASE']  = '/content/drive/MyDrive/VQA_ML/AVA_VizWiz'  # if your Drive layout differs

def sh(args, check=False):
    print('$', ' '.join(args))
    proc = subprocess.run(args, text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f'command failed ({proc.returncode}): {args}')
    return proc

# 1) Clone or update to the latest push on main
if not os.path.exists(REPO_ROOT):
    sh(['git', 'clone', '--depth', '1', REPO_URL, REPO_ROOT], check=True)
else:
    if sh(['git', '-C', REPO_ROOT, 'pull', '--ff-only']).returncode != 0:
        print('[setup] pull failed; hard-resetting the (disposable) clone to origin/main')
        sh(['git', '-C', REPO_ROOT, 'fetch', 'origin', 'main'], check=True)
        sh(['git', '-C', REPO_ROOT, 'reset', '--hard', 'origin/main'], check=True)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
head = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'[setup] repo at {REPO_ROOT}, HEAD={head}')

# 2) Install ONLY what is missing (import-probe first, zero pip on warm runtimes)
NEEDED = {  # import name -> pip spec
    'torch': 'torch', 'torchvision': 'torchvision',
    'numpy': 'numpy', 'pandas': 'pandas', 'pyarrow': 'pyarrow',
    'sklearn': 'scikit-learn', 'scipy': 'scipy',
    'matplotlib': 'matplotlib', 'PIL': 'Pillow', 'tqdm': 'tqdm',
    'transformers': 'transformers>=4.44',
    'open_clip': 'open-clip-torch>=2.26',
    'timm': 'timm>=1.0',
    'einops': 'einops', 'ftfy': 'ftfy', 'regex': 'regex',
}
missing = [spec for mod, spec in NEEDED.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print('[setup] installing missing packages:', missing)
    sh([sys.executable, '-m', 'pip', 'install', '-q', '--progress-bar', 'off',
        *missing], check=True)
else:
    print('[setup] all packages already present - no pip work needed.')

# 3) Safety net: verify the compiled numeric stack imports; repair only if broken
r = sh([sys.executable, 'scripts/colab_preflight.py'])
if r.returncode == 10:
    raise SystemExit('Numeric stack was repaired. Runtime -> Restart runtime, '
                     'then rerun this SETUP cell before continuing.')
if r.returncode != 0:
    raise RuntimeError('Numeric stack check failed - see output above.')

print('[setup] READY. Run the next cells - finished experiments skip themselves.')


## E6b - BLIP confidence harvest  *(GPU - A100 fastest)*
Second frozen VQA model. Confidence = geometric mean of generated-token
probabilities (greedy decoding). Cached to Drive, resumes from shards
after a crash. Stages val+train images if this is a fresh runtime.

In [ ]:
# ====================================================================
#  E6b - BLIP-VQA-base confidence harvest
#  RUNTIME: GPU required (A100 > L4 > T4). Refuses to crawl on CPU.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e6b_vqaconf_blip
e6b_vqaconf_blip.main()


## E7d - Selective-prediction diagnostics on BLIP  *(CPU - fine to run in the same GPU session right after E6b)*
Reruns the entire E7b/E7c method battery + the GT-defect oracle against
BLIP confidence, applies BH-FDR, and computes refusal-gated ARR/FRR at
90% target coverage with the BLIP gate.

In [ ]:
# ====================================================================
#  E7d - BLIP selective-prediction diagnostics
#  RUNTIME: CPU. Cached data + E6b parquet only.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e7d_blip_diagnostics
e7d_blip_diagnostics.main()


## E8d - Regenerate F8 with the full method battery  *(CPU, seconds)*
Rebuilds `F8_selective_diagnostics` from the E7b+E7c summaries and adds
the BLIP panel automatically once E7d has run. Copy the new PDF into
`IEEE Access template/figures/` afterwards. Safe to rerun anytime.

In [ ]:
# ====================================================================
#  E8d - F8 full-battery regeneration
#  RUNTIME: CPU. Reads cached summaries only.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e8d_figures
e8d_figures.main()
